# Jupyter, Markdown, and Git

This notebook is a short, runnable tour of the tools that sit underneath every other project in
this repository: the interactive notebook, Markdown for explanation, Git for version control, and
the habits that make an analysis reproducible. It downloads nothing, so it runs offline.

## Learning objectives

By the end of this notebook you should be able to:

- Explain the difference between a code cell and a Markdown cell, and why execution order matters.
- Write the Markdown constructs most often used in technical notes.
- Run the basic Git loop — `init`, `status`, `add`, `commit`, `log` — inside a temporary
  directory without touching your real project.
- Describe why seeds, pinned versions, deterministic ordering, and provenance make a result
  reproducible.

## Concept

### A notebook is a list of cells

A notebook is an ordered sequence of cells, each of one of two kinds.

- **Code cells** are executed by a kernel. They share one namespace, so a variable defined in an
  early cell is available in a later one.
- **Markdown cells** are formatted text. They are not executed; they carry the explanation.

Because all code cells share a kernel, **execution order is part of the result**. A notebook that
was run out of order can show output that cannot be reproduced by *Restart Kernel and Run All*.
The remedy is a habit, not a feature: restart and run everything from the top before sharing.

### Markdown in one table

| Construct | Syntax | Typical use |
|---|---|---|
| Heading | `#`, `##`, `###` | Structure |
| Emphasis | `**bold**`, `*italic*` | Highlighting |
| Inline code | `` `name` `` | Identifiers, paths, commands |
| Fenced block | triple backticks | Multi-line code |
| List | `-` or `1.` | Bullets and steps |
| Link | `[text](url)` | References |
| Table | pipes and dashes | Comparisons |
| Maths | `$...$` or `$$...$$` | Formulas |

### Git records snapshots

Git stores snapshots of a project called **commits**. The everyday loop is:

```text
git status            # what changed?
git add <files>       # stage what belongs in the next snapshot
git commit -m "..."   # record the snapshot
git log --oneline     # review history
```

A **branch** is a movable name for a line of work. Creating one lets you experiment without
disturbing the main line, and merging brings the work back when it is ready.

### Reproducibility is a workflow property

A result is reproducible when someone else, using the same code and data, gets the same answer.
Four habits do most of the work:

1. **Fixed seeds** for anything random.
2. **Pinned dependency versions**, recorded in `requirements.txt`.
3. **Deterministic ordering** for sorting, grouping, and file iteration.
4. **Recorded provenance**: where the data came from, when, and its licence.

## Worked example

We will do three small things: summarise a tiny in-memory table, demonstrate the difference
between raw Markdown and its rendered form, and run a Git workflow in a throwaway directory. The
dataset is deliberately trivial — the tools, not the data, are the subject.

### A tiny table, summarised in Python

In [1]:
import statistics

penguins = [
    {"species": "Adelie", "mass_g": 3750},
    {"species": "Chinstrap", "mass_g": 3800},
    {"species": "Gentoo", "mass_g": 5000},
]

for row in penguins:
    print(f"{row['species']:<10} {row['mass_g']:>5} g")

mean_mass = statistics.mean(row["mass_g"] for row in penguins)
print(f"\nmean mass: {mean_mass:.1f} g")

Adelie      3750 g
Chinstrap   3800 g
Gentoo      5000 g

mean mass: 4183.3 g


### Markdown is text that renders

A Markdown cell is stored as plain text. The cell below prints the raw source of a small table so
you can see the syntax; in a real Markdown cell the same text would render as a table.

In [2]:
raw_markdown = """| Species   | Mass (g) |
|-----------|---------:|
| Adelie    |     3750 |
| Chinstrap |     3800 |
| Gentoo    |     5000 |"""

print(raw_markdown)

| Species   | Mass (g) |
|-----------|---------:|
| Adelie    |     3750 |
| Chinstrap |     3800 |
| Gentoo    |     5000 |


### A seed fixes a random workflow

The next cell draws five species names at random, twice. Both runs use the seed `42`, so the two
lists are identical. Change the seed to `7` and the lists change — but they still agree with each
other, because the seed is doing its job.

The repository's shared helper `ds_practice.set_seed` seeds the standard-library and NumPy random
number generators at once, which is why we use it here instead of seeding each library by hand.

In [3]:
import random

from ds_practice import set_seed

set_seed(42)

rng = random.Random(42)
first = [rng.choice(penguins)["species"] for _ in range(5)]

rng = random.Random(42)
second = [rng.choice(penguins)["species"] for _ in range(5)]

print("run 1:   ", first)
print("run 2:   ", second)
print("identical:", first == second)

run 1:    ['Gentoo', 'Adelie', 'Adelie', 'Gentoo', 'Chinstrap']
run 2:    ['Gentoo', 'Adelie', 'Adelie', 'Gentoo', 'Chinstrap']
identical: True


### Git in a temporary directory

The next cell creates a brand-new directory with `tempfile.mkdtemp()`, initialises a repository
there, makes a commit, and then tries a short branch. Nothing outside that directory is touched.

If Git is not installed, the cell prints a short explanation instead of failing, so the notebook
still completes offline on a machine without Git.

In [4]:
import shutil
import subprocess
import tempfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="git-demo-"))
print("temporary directory:", workdir)

if shutil.which("git") is None:
    print("Git is not installed; skipping the demonstration.")
    print("Install Git and re-run this cell to see init/add/commit/log/branch in action.")
else:
    git = [
        "git",
        "-c", "user.name=Notebook Demo",
        "-c", "user.email=demo@example.com",
        "-C", str(workdir),
    ]

    def run(*args):
        result = subprocess.run(git + list(args), capture_output=True, text=True)
        if result.returncode != 0:
            print("git", *args, "->", result.stderr.strip())
        return result.stdout.strip()

    run("init", "-q")
    (workdir / "notes.txt").write_text("first draft\n")
    print("\nstatus after adding a file:")
    print(run("status", "--short"))

    run("add", "notes.txt")
    run("commit", "-q", "-m", "Add notes")
    default_branch = run("rev-parse", "--abbrev-ref", "HEAD")
    print(f"\nlog on the default branch ({default_branch}):")
    print(run("log", "--oneline"))

    run("switch", "-c", "experiment")
    (workdir / "notes.txt").write_text("first draft\nsecond line on the branch\n")
    run("add", "notes.txt")
    run("commit", "-q", "-m", "Extend notes on the branch")
    print("\nlog on the experiment branch:")
    print(run("log", "--oneline"))

    run("switch", "-q", default_branch)
    print(f"\nlog back on {default_branch} (branch commit is absent):")
    print(run("log", "--oneline"))

temporary directory: /tmp/git-demo-g7wxemy2

status after adding a file:
?? notes.txt

log on the default branch (master):
b7a971c Add notes

log on the experiment branch:
69841e3 Extend notes on the branch
b7a971c Add notes

log back on master (branch commit is absent):
b7a971c Add notes


## Exercises

1. Add a Markdown cell to this notebook with a level-2 heading, a three-item bulleted list, one
   inline code span, and a link to `docs/datasets.md`. Then add a code cell that prints the
   current working directory with `pathlib.Path.cwd()`, and explain in one sentence why that path
   can differ between machines.
2. Extend the temporary Git repository so that the `experiment` branch has a *separate* commit
   that the default branch does not, then switch between branches and print `git log --oneline`
   for each. Write one sentence explaining why the branch commit is invisible on the default
   branch until the branches are merged.
3. Change the seed in the sampling cell from `42` to `7` and run the cell twice. Note what stays
   the same and what changes, then write one sentence on why a fixed seed is necessary but not
   sufficient for reproducibility.

## Limitations

- This is a teaching notebook, not a complete Git or Jupyter manual. Remotes, rebasing, and merge
  conflicts are out of scope.
- Notebook front ends differ: JupyterLab, the classic Notebook, VS Code, and other editors expose
  different menus and shortcuts for the same actions.
- The Git demonstration detects the default branch name (for example `main` or `master`) rather
  than assuming one, because the default is configurable.
- A fixed seed makes a single workflow repeatable, but the four habits together — seed, pinned
  versions, deterministic ordering, and provenance — are what make a result genuinely portable.